# Micrograd — full working reference notebook

This notebook implements Karpathy's micrograd from scratch, with **real runnable code** and visualizations at every step. Type along — do not just run cells.

**What you'll build:**
1. A `Value` class wrapping a scalar with autograd
2. A computation-graph visualizer (uses graphviz if installed, falls back to a matplotlib rendering)
3. A `Neuron`, `Layer`, `MLP`
4. Training on a tiny 2D dataset with a real loss curve plot

**Install (once):**
```bash
pip install numpy matplotlib graphviz
# graphviz python package needs the system binary too:
#   macOS: brew install graphviz
#   Ubuntu: sudo apt install graphviz
```

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(1337)
np.random.seed(1337)

## Step 1 — the Value class

Each `Value` wraps a scalar. It remembers its parents (`_prev`), the op that produced it (`_op`), and its gradient. `_backward` is a closure that propagates the gradient from this node to its parents using the chain rule.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), 'only supports int/float powers for now'
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t*t) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    # dunder goodies
    def __neg__(self):         return self * -1
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * (other ** -1 if isinstance(other, Value) else Value(other) ** -1)

    def backward(self):
        # topological order: children before parents, parents first reversed
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

### Sanity test

Compute `L = (a * b + b ** 3) * 2` and verify gradients by hand:

- dL/da = 2 · b = 6
- dL/db = 2 · (a + 3·b²) = 2 · (-2 + 27) = 50

In [ ]:
a = Value(-2.0, label='a')
b = Value( 3.0, label='b')
c = a * b;           c.label = 'c = a*b'
d = b ** 3;          d.label = 'd = b^3'
e = c + d;           e.label = 'e = c+d'
L = e * 2;           L.label = 'L = e*2'
L.backward()
print('a:', a)   # grad should be 6
print('b:', b)   # grad should be 50
print('L:', L)   # data should be (-6 + 27) * 2 = 42

## Step 2 — visualize the computation graph

This is the payoff cell. See the graph you just built.

In [ ]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_graph(root):
    """Render the computation graph. Uses graphviz if available; else matplotlib."""
    try:
        from graphviz import Digraph
        dot = Digraph(format='svg', graph_attr={'rankdir': 'LR', 'bgcolor': '#F5EEDC'})
        nodes, edges = trace(root)
        for n in nodes:
            uid = str(id(n))
            label = f"{{ {n.label} | data {n.data:.3f} | grad {n.grad:.3f} }}"
            dot.node(uid, label=label, shape='record',
                     style='filled', fillcolor='#FFF9E9', color='#8B5E3C', fontname='Inter')
            if n._op:
                op_id = uid + n._op
                dot.node(op_id, label=n._op, shape='circle',
                         style='filled', fillcolor='#B8892E', fontcolor='#FFF9E9', fontname='JetBrains Mono')
                dot.edge(op_id, uid, color='#8B5E3C')
        for a, b in edges:
            dot.edge(str(id(a)), str(id(b)) + b._op, color='#8B5E3C')
        return dot
    except ImportError:
        # matplotlib fallback
        nodes, edges = trace(root)
        nodes = list(nodes)
        # simple layered layout via BFS depth
        depth = {n: 0 for n in nodes}
        changed = True
        while changed:
            changed = False
            for a, b in edges:
                if depth[b] < depth[a] + 1:
                    depth[b] = depth[a] + 1
                    changed = True
        by_depth = {}
        for n in nodes: by_depth.setdefault(depth[n], []).append(n)
        pos = {}
        for d, ns in by_depth.items():
            for i, n in enumerate(ns):
                pos[n] = (d, i - (len(ns)-1)/2)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.set_facecolor('#F5EEDC')
        for a, b in edges:
            ax.annotate('', xy=pos[b], xytext=pos[a],
                        arrowprops=dict(arrowstyle='->', color='#8B5E3C', lw=1.2))
        for n, (x, y) in pos.items():
            label = f"{n.label}\n{n.data:.2f}|{n.grad:.2f}"
            ax.text(x, y, label, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF9E9', edgecolor='#8B5E3C'))
        ax.set_xlim(-0.5, max(depth.values()) + 0.5)
        ax.set_ylim(-3, 3)
        ax.set_axis_off()
        plt.tight_layout()
        return fig

draw_graph(L)

## Step 3 — Neuron, Layer, MLP

A neuron = dot product + bias + activation. Everything else is just more of them.

In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    def parameters(self):
        return []

class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act
    def parameters(self):
        return self.w + [self.b]

class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=(i != len(nouts)-1))
                       for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

mlp = MLP(3, [4, 4, 1])
print(f'MLP has {len(mlp.parameters())} parameters')

## Step 4 — train on a toy dataset

Four examples. Predict binary target. Watch the loss curve.

In [ ]:
xs = [
    [ 2.0,  3.0, -1.0],
    [ 3.0, -1.0,  0.5],
    [ 0.5,  1.0,  1.0],
    [ 1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

losses = []
for step in range(200):
    preds = [mlp(x) for x in xs]
    loss = sum(((p - y) ** 2 for p, y in zip(preds, ys)), Value(0.0))
    mlp.zero_grad()
    loss.backward()
    for p in mlp.parameters():
        p.data -= 0.05 * p.grad
    losses.append(loss.data)
    if step % 20 == 0:
        print(f'step {step:3d} | loss {loss.data:.4f}')

print('\nfinal predictions:', [round(p.data, 3) for p in preds])
print('targets:           ', ys)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor('#F5EEDC')
ax.set_facecolor('#FFF9E9')
ax.plot(losses, color='#8B5E3C', lw=2)
ax.set_title('Loss over training steps', fontsize=13, color='#2A221B')
ax.set_xlabel('step', color='#7A6B5D')
ax.set_ylabel('loss (MSE)', color='#7A6B5D')
ax.grid(alpha=0.3, color='#D4C4B0')
ax.set_yscale('log')
for spine in ax.spines.values(): spine.set_color('#D4C4B0')
plt.tight_layout()
plt.show()

## Step 5 — see the graph of a single training example

Now that the network is trained, pull out just the first example's loss contribution and look at it.

In [ ]:
p0 = mlp(xs[0])
ex_loss = (p0 - ys[0]) ** 2
ex_loss.backward()
# Note: the graph for a deep MLP is BIG — try with nouts=[2, 1] if you want a readable one
draw_graph(ex_loss)

## You did it

You just wrote PyTorch. With ~50 lines of pure Python.

Go tick the checklist in [[../01-micrograd/index|4.1 Micrograd]] and move to the next video.

**Harder follow-ups if you want them** (ask the Code Tutor for hints, not answers):
- Add `__getitem__`, `log`, and implement softmax + cross-entropy on top
- Implement an `Optimizer` class (SGD, then Adam)
- Vectorize the Neuron with numpy — feel how much faster it gets